# NVS Benchmark no Google Colab

[Abrir no Colab](https://colab.research.google.com/github/PedroHeinrichSP/TCC-Source-Code/blob/update/notebooks/nvs_benchmark_colab.ipynb) — clique para abrir e rodar o notebook no Google Colab.

Notebook focado em execução no Colab: clone, setup, benchmark rápido (ou metrics-only de artifacts pré-treinados), relatório HTML, download e backup opcional no Drive.


## O que editar e onde

| O que mudar | Onde | Como |
| --- | --- | --- |
| **URL do repositório ou branch** | Célula 2 | Mude `REPO_URL` ou `BRANCH` |
| **Ambiente (para testes paralelos)** | Célula 2 | Mude `ENVIRONMENT` (ex: "colab", "local_test1", etc) |
| **Usar artifacts pré-treinados** | Célula 2 | Mude `SKIP_TRAINING = True` |
| **Método (nerf_static, gs_static, etc)** | Célula 2 | Mude `SELECTED_METHOD` |
| **Dataset (blender_synthetic, d_nerf, etc)** | Célula 2 | Mude `SELECTED_DATASET` |
| **Preset (smoke, quick, preview, standard, full)** | Célula 2 | Mude `SELECTED_PRESET` |
| **Modo de execução (quick_check ou full)** | Célula 2 | Mude `RUN_MODE` |
| **Modo estrito (resultados validados)** | Célula 2 | Mude `STRICT_RESULTS` |
| **Backup no Google Drive** | Célula 6 | Mude `USE_GOOGLE_DRIVE = True` ou `False` |
| **Gerar PDF do relatório** | Célula 2 | Mude `GENERATE_PDF` |
| **Upload de artifacts** | Célula 7 | Siga as instruções (ZIP com checkpoint.pth + renders/) |

## Fluxo de execução

### Modo Full Training (SKIP_TRAINING=False)
```
[Clone] -> [Setup] -> [GPU?] -> [Drive?] -> [Datasets] -> [Full training+infer+metrics] -> [Relatorio] -> [Download]
```

### Modo Metrics-Only (SKIP_TRAINING=True)
```
[Clone] -> [Setup] -> [GPU?] -> [Drive?] -> [Upload artifacts] -> [Recompute metrics] -> [Relatorio] -> [Download]
```

**Tempo estimado**:
- Full training: 10–30 min (depende do preset e dataset)
- Metrics-only: 2–5 min (recompute apenas)

In [ ]:
# Parametros globais e funções auxiliares
import json
import os
import shutil
import sys
import zipfile
import subprocess
from pathlib import Path
from typing import Optional

# ============================================================================
# FUNÇÕES AUXILIARES (definidas aqui para usar antes do clone)
# ============================================================================

def generate_run_id(environment: str, preset: str, method: str, dataset: str) -> str:
    """Gera RUN_ID automaticamente: {environment}_{preset}_{method}_{dataset}"""
    dataset_clean = dataset.split("/")[-1].replace(" ", "_").replace("-", "_").lower()
    return f"{environment}_{preset}_{method}_{dataset_clean}"


def extract_zip_artifacts(zip_path: str, extract_to: str) -> tuple:
    """Extrai checkpoint e renders_dir de um arquivo ZIP."""
    extract_path = Path(extract_to)
    extract_path.mkdir(parents=True, exist_ok=True)

    print(f"Extraindo: {zip_path}")
    print(f"Destino:   {extract_to}")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"✓ Extração concluída")

    checkpoint_patterns = ["checkpoint.pth", "checkpoint.pkl", "checkpoint.ckpt", "checkpoint.pt"]
    checkpoint_patterns += ["model.pth", "model.pkl", "final.pth", "final.ckpt"]
    checkpoint_path = None

    for pattern in checkpoint_patterns:
        matches = list(extract_path.glob(f"**/{pattern}"))
        if matches:
            checkpoint_path = str(matches[0])
            print(f"✓ Checkpoint encontrado: {Path(checkpoint_path).name}")
            break

    if not checkpoint_path:
        for ext in ["*.pth", "*.pkl", "*.ckpt", "*.pt"]:
            matches = list(extract_path.glob(f"**/{ext}"))
            if matches:
                checkpoint_path = str(matches[0])
                print(f"✓ Checkpoint encontrado: {Path(checkpoint_path).name}")
                break

    if not checkpoint_path:
        raise FileNotFoundError("Nenhum arquivo de checkpoint encontrado no ZIP")

    renders_dir = None
    render_candidates = ["renders", "output", "images", "images_val"]
    for candidate in render_candidates:
        candidate_path = extract_path / candidate
        if candidate_path.exists() and candidate_path.is_dir():
            pngs = list(candidate_path.glob("*.png"))
            if pngs:
                renders_dir = str(candidate_path)
                print(f"✓ Renders encontrados: {candidate}/ ({len(pngs)} frames PNG)")
                break

    if not renders_dir:
        raise FileNotFoundError("Nenhum diretório com arquivos PNG encontrado no ZIP")

    return checkpoint_path, renders_dir


def get_selection_state_file(use_google_drive: bool, drive_file: str, local_file: str) -> Path:
    """Usa Drive quando estiver montado; senão usa arquivo local do runtime."""
    drive_ready = use_google_drive and Path("/content/drive/MyDrive").exists()
    return Path(drive_file if drive_ready else local_file)


def load_selection_state(state_file: Path, defaults: dict) -> dict:
    """Carrega selecão salva de um arquivo JSON."""
    if not state_file.exists():
        return defaults.copy()

    try:
        state = json.loads(state_file.read_text(encoding="utf-8"))
        result = defaults.copy()
        result.update(state)
        return result
    except Exception as exc:
        print(f"[warn] Falha ao ler estado salvo em {state_file}: {exc}")
        return defaults.copy()


def save_selection_state(state_file: Path, state: dict, reason: str = "manual") -> None:
    """Persiste estado atual para reutilização nas próximas execuções."""
    state_file.parent.mkdir(parents=True, exist_ok=True)
    payload = {**state, "saved_reason": reason}
    state_file.write_text(json.dumps(payload, ensure_ascii=True, indent=2), encoding="utf-8")


def run_logged(cmd, label: str, check: bool = True, cwd: Optional[str] = None):
    """Executa um comando e imprime stdout/stderr para facilitar depuração."""
    print("\n" + "=" * 70)
    print(f"[{label}] $ {' '.join(map(str, cmd))}")
    print("=" * 70)
    result = subprocess.run(
        [str(part) for part in cmd],
        text=True,
        capture_output=True,
        cwd=str(cwd) if cwd else None,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Comando '{label}' falhou com code={result.returncode}")
    return result


def sync_dir_if_exists(src, dst) -> bool:
    """Copia um diretório existente de src para dst e retorna True."""
    src = Path(src)
    dst = Path(dst)
    if not src.exists() or not src.is_dir():
        return False
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    return True


def _collapse_duplicate_segments(path: Path) -> Path:
    """Remove segmentos consecutivos repetidos em um caminho."""
    parts = list(path.parts)
    if not parts:
        return path
    collapsed = [parts[0]]
    for part in parts[1:]:
        if part != collapsed[-1]:
            collapsed.append(part)
    return Path(*collapsed)


def normalize_dataset_path(path: Path) -> Path:
    """Normaliza a raiz do dataset para o diretório que contém transforms_train.json."""
    path = Path(path).expanduser().resolve()
    if (path / "transforms_train.json").exists():
        return path

    collapsed_path = _collapse_duplicate_segments(path)
    if (collapsed_path / "transforms_train.json").exists():
        return collapsed_path.resolve()

    nested_matches = list(collapsed_path.glob("**/transforms_train.json"))
    if nested_matches:
        return nested_matches[0].parent.resolve()
    return collapsed_path.resolve()


def discover_available_datasets(use_google_drive: bool, drive_data_dir: str) -> dict:
    """Descobre datasets disponíveis no runtime local e, se habilitado, no Drive."""
    discovered = {}
    search_roots = [Path("./data")]
    if use_google_drive:
        search_roots.extend([Path(drive_data_dir), Path("/content/drive/MyDrive/NVS_Benchmark/data")])

    for base in search_roots:
        if not base.exists():
            continue
        for train_file in base.glob("**/transforms_train.json"):
            dataset_root = normalize_dataset_path(train_file.parent)
            try:
                relative_root = dataset_root.resolve().relative_to(base.resolve())
                dataset_key = relative_root.parts[0] if relative_root.parts else dataset_root.name
            except Exception:
                dataset_key = dataset_root.name
            if dataset_key == "nerf_synthetic":
                dataset_key = "blender_synthetic"
            if dataset_key not in discovered or len(str(dataset_root)) < len(discovered[dataset_key]):
                discovered[dataset_key] = str(dataset_root)
    return discovered


# ============================================================================
# PARÃ‚METROS GLOBAIS
# ============================================================================

REPO_URL = "https://github.com/PedroHeinrichSP/TCC-Source-Code.git"
REPO_DIR = "/content/TCC"
BRANCH = "update"
ENVIRONMENT = "colab"
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NVS_Benchmark"
DRIVE_DATA_DIR = f"{DRIVE_OUTPUT_DIR}/data"
DRIVE_ARTIFACTS_DIR = f"{DRIVE_OUTPUT_DIR}/artifacts"
CATALOG_FILE = "./src/configs/install_catalog.json"

LOAD_SAVED_SELECTION = False
SELECTED_METHOD = "gs_dynamic"
SELECTED_DATASET = "d_nerf"
SELECTED_PRESET = "preview"
RUN_MODE = "full"
APPLY_COMPATIBILITY_FILTER = True
STRICT_RESULTS = True
GENERATE_PDF = True
MIN_REQUIRED_PAIRS = 1
SELECTED_DATASET_ROOT = "/content/TCC/data/d_nerf/lego"

SKIP_TRAINING = False
UPLOADED_CHECKPOINT = None
UPLOADED_RENDERS_DIR = None
UPLOADED_TRAIN_SECONDS = 0.0
UPLOADED_INFERENCE_SECONDS = 0.0

RUN_ID = None

LOCAL_SELECTION_STATE_FILE = "/content/nvs_benchmark_selection.json"
DRIVE_SELECTION_STATE_FILE = f"{DRIVE_OUTPUT_DIR}/selection_state.json"

# ============================================================================
# Carregar estado salvo (se houver)
# ============================================================================
state_defaults = {
    "selected_method": SELECTED_METHOD,
    "selected_dataset": SELECTED_DATASET,
    "selected_preset": SELECTED_PRESET,
    "run_mode": RUN_MODE,
    "apply_compatibility_filter": APPLY_COMPATIBILITY_FILTER,
    "strict_results": STRICT_RESULTS,
    "generate_pdf": GENERATE_PDF,
    "min_required_pairs": MIN_REQUIRED_PAIRS,
}

loaded_from_state = False
if LOAD_SAVED_SELECTION:
    state_file = get_selection_state_file(USE_GOOGLE_DRIVE, DRIVE_SELECTION_STATE_FILE, LOCAL_SELECTION_STATE_FILE)
    loaded_state = load_selection_state(state_file, state_defaults)
    SELECTED_METHOD = loaded_state["selected_method"]
    SELECTED_DATASET = loaded_state["selected_dataset"]
    SELECTED_PRESET = loaded_state["selected_preset"]
    RUN_MODE = loaded_state["run_mode"]
    APPLY_COMPATIBILITY_FILTER = loaded_state["apply_compatibility_filter"]
    STRICT_RESULTS = loaded_state["strict_results"]
    GENERATE_PDF = loaded_state["generate_pdf"]
    MIN_REQUIRED_PAIRS = loaded_state["min_required_pairs"]
    loaded_from_state = state_file.exists()

print("=" * 70)
print("SELEÇÃO INICIAL")
print("=" * 70)
print(f"Ambiente:         {ENVIRONMENT}")
print(f"Método:           {SELECTED_METHOD}")
print(f"Dataset:          {SELECTED_DATASET}")
print(f"Preset:           {SELECTED_PRESET}")
print(f"Modo:             {RUN_MODE}")
print(f"Estrito:          {STRICT_RESULTS}")
print(f"Gerar PDF:        {GENERATE_PDF}")
print(f"Min pairs:        {MIN_REQUIRED_PAIRS}")
print(f"Estado carregado: {'SIM' if loaded_from_state else 'NÃO (priorizando valores editados nesta célula)'}")
print("=" * 70)

# Salvar estado atual
state_file = get_selection_state_file(USE_GOOGLE_DRIVE, DRIVE_SELECTION_STATE_FILE, LOCAL_SELECTION_STATE_FILE)
current_state = {
    "selected_method": SELECTED_METHOD,
    "selected_dataset": SELECTED_DATASET,
    "selected_preset": SELECTED_PRESET,
    "run_mode": RUN_MODE,
    "apply_compatibility_filter": APPLY_COMPATIBILITY_FILTER,
    "strict_results": STRICT_RESULTS,
    "generate_pdf": GENERATE_PDF,
    "min_required_pairs": MIN_REQUIRED_PAIRS,
}
save_selection_state(state_file, current_state, reason="cell2-applied")

ModuleNotFoundError: No module named 'colab_utils'

In [ ]:
# Clone do repositório
# Configurações estão na Célula 2: REPO_URL, BRANCH, REPO_DIR
print("=" * 70)
print(f"Clonando repositório: {REPO_URL} (branch: {BRANCH})")
print("=" * 70)

import os
import shutil
import subprocess
import sys

try:
    __import__("google.colab")
except Exception as exc:
    raise RuntimeError("Este notebook foi desenhado para Google Colab.") from exc

# Mude para um diretório seguro antes de remover REPO_DIR (evita erros no Colab)
os.chdir("/content") if os.path.exists("/content") else os.chdir(os.path.expanduser("~"))

if os.path.exists(REPO_DIR):
    print(f"Removendo pasta existente: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

print("Clonando...")
clone_result = subprocess.run([
    "git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR
], text=True, capture_output=True)
if clone_result.stdout:
    print(clone_result.stdout)
if clone_result.stderr:
    print(clone_result.stderr)
if clone_result.returncode != 0:
    raise RuntimeError(f"Falha ao clonar o repositório (code={clone_result.returncode}).")

os.chdir(REPO_DIR)
print(f"\n✓ Projeto clonado em: {os.getcwd()}")

Clonando repositório: https://github.com/PedroHeinrichSP/TCC-Source-Code.git (branch: update)


RuntimeError: Este notebook foi desenhado para Google Colab.

In [ ]:
# Setup do ambiente
print("=" * 70)
print("Instalando dependências...")
print("=" * 70)

pip_upgrade = subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], text=True, capture_output=True)
print(pip_upgrade.stdout)
if pip_upgrade.stderr:
    print(pip_upgrade.stderr)
if pip_upgrade.returncode != 0:
    raise RuntimeError(f"Falha ao atualizar pip (code={pip_upgrade.returncode}).")

pip_install = subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], text=True, capture_output=True)
print(pip_install.stdout)
if pip_install.stderr:
    print(pip_install.stderr)
if pip_install.returncode != 0:
    raise RuntimeError(f"Falha ao instalar o projeto em modo editável (code={pip_install.returncode}).")

print("\nVerificando instalação...")
status_result = subprocess.run([sys.executable, "-m", "nvs_benchmark.cli", "status"], text=True, capture_output=True)
print(status_result.stdout)
if status_result.stderr:
    print(status_result.stderr)
if status_result.returncode != 0:
    raise RuntimeError(f"Falha na verificação de status (code={status_result.returncode}).")

# ============================================================================
# Clone dos métodos necessários (D-NeRF é requerido mesmo para nerf_static)
# ============================================================================
print("\n" + "=" * 70)
print("Clonando repositórios de métodos...")
print("=" * 70)

from pathlib import Path

# Garantir que third_party existe
third_party_dir = Path("./third_party")
third_party_dir.mkdir(parents=True, exist_ok=True)
print(f"✓ Diretório third_party pronto: {third_party_dir.absolute()}")

methods_to_clone = [
    {
        "id": "d_nerf",
        "path": "./third_party/d_nerf",
        "url": "https://github.com/albertpumarola/D-NeRF.git",
        "description": "D-NeRF (suporte temporal para NeRF)",
    },
    {
        "id": "gaussian_splatting",
        "path": "./third_party/gaussian_splatting",
        "url": "https://github.com/graphdeco-inria/gaussian-splatting.git",
        "description": "3D Gaussian Splatting",
    },
]

cloned_methods = []
for method in methods_to_clone:
    method_path = Path(method["path"])
    if method_path.exists() and list(method_path.iterdir()):
        print(f"✓ {method['id']} já existe em {method['path']}")
        cloned_methods.append(method['id'])
    else:
        print(f"\nClonando {method['description']}...")
        print(f"  URL: {method['url']}")
        print(f"  Destino: {method['path']}")

        # Garantir que o diretório pai existe
        method_path.parent.mkdir(parents=True, exist_ok=True)

        # Prefer cloning with submodules; fallback to shallow clone if recursive fails
        clone_cmd_recursive = ["git", "clone", "--depth", "1", "--recursive", method["url"], method["path"]]
        clone_cmd = ["git", "clone", "--depth", "1", method["url"], method["path"]]
        print(f"$ {' '.join(clone_cmd_recursive)}")
        clone_result = subprocess.run(clone_cmd_recursive, text=True, capture_output=True)
        if clone_result.returncode != 0:
            print("[warn] Recursive clone falhou, tentando clone simples...")
            print(f"$ {' '.join(clone_cmd)}")
            clone_result = subprocess.run(clone_cmd, text=True, capture_output=True)

        if clone_result.stdout:
            print(clone_result.stdout)

        if clone_result.returncode == 0:
            # Validar que o clone funcionou
            if method_path.exists() and list(method_path.iterdir()):
                print(f"✓ {method['id']} clonado com sucesso")
                cloned_methods.append(method['id'])
            else:
                print(f"⚠ Clone falhou (diretório vazio ou não acessível): {method['path']}")
                if clone_result.stderr:
                    print(f"  Erro: {clone_result.stderr}")
        else:
            print(f"⚠ Falha ao clonar {method['id']} (returncode={clone_result.returncode})")
            if clone_result.stderr:
                print(f"  Erro: {clone_result.stderr}")
            print(f"  (Você pode clonar manualmente depois se necessário)")

# ============================================================================
# Instalar dependências compiladas (Gaussian Splatting no Colab)
# ============================================================================
print("\n" + "=" * 70)
print("Instalando dependências compiladas dos métodos...")
print("=" * 70)

def install_python_runtime_deps(label: str, packages: list[str]) -> bool:
    install_cmd = [sys.executable, "-m", "pip", "install", *packages]
    print(f"\nInstalando dependencias Python de runtime para {label}: {', ' .join(packages)}")
    print(f"$ {' '.join(install_cmd)}")
    result = subprocess.run(install_cmd, text=True, capture_output=True)
    if result.stdout:
        for line in result.stdout.split("\n")[-40:]:
            if line.strip():
                print(line)
    if result.returncode == 0:
        print(f"OK: dependencias Python de {label} instaladas")
        return True
    print(f"AVISO: falha ao instalar dependencias Python de {label} (code={result.returncode})")
    if result.stderr:
        for line in result.stderr.split("\n")[-30:]:
            if line.strip():
                print(f"  {line}")
    return False

gs_path = Path("./third_party/gaussian_splatting")
gs_extensions_ok = False
if gs_path.exists():
    print(f"\nGaussian Splatting encontrado em: {gs_path}")
    # Mostrar submódulos/status para diagnóstico
    try:
        submod = subprocess.run(["git", "-C", str(gs_path), "submodule", "status"], text=True, capture_output=True)
        if submod.stdout:
            print("[git submodule status]\n" + submod.stdout)
    except Exception:
        pass

    print("[info] Repositório base do Gaussian Splatting presente. Ele não é um pacote pip instalável.")
    runtime_deps_ok = install_python_runtime_deps("gaussian-splatting", ["plyfile>=1.0.3", "joblib>=1.4"])

    extension_targets = [
        ("diff-gaussian-rasterization", gs_path / "submodules" / "diff-gaussian-rasterization", True),
        ("simple-knn", gs_path / "submodules" / "simple-knn", True),
        ("fused-ssim", gs_path / "submodules" / "fused-ssim", False),
    ]
    gs_extensions_ok = runtime_deps_ok
    for label, package_path, required in extension_targets:
        if not package_path.exists():
            level = "warn" if required else "info"
            print(f"[{level}] {label}: submódulo ausente em {package_path}")
            if required:
                gs_extensions_ok = False
            continue

        if label == "simple-knn":
            package_dir = package_path / "simple_knn"
            package_dir.mkdir(parents=True, exist_ok=True)
            init_file = package_dir / "__init__.py"
            if not init_file.exists():
                init_file.write_text("", encoding="utf-8")
                print(f"[info] {label}: criado arquivo de pacote em {init_file}")

        install_cmd = [sys.executable, "-m", "pip", "install", "--no-build-isolation", "-e", str(package_path)]
        print(f"\nInstalando extensão {label} em: {package_path}")
        print(f"$ {' '.join(install_cmd)}")
        install_result = subprocess.run(install_cmd, text=True, capture_output=True)
        if install_result.stdout:
            for line in install_result.stdout.split('\n')[-40:]:
                if line.strip():
                    print(line)
        if install_result.returncode == 0:
            print(f"OK: {label} instalado com sucesso")
        else:
            print(f"AVISO: falha ao instalar {label} (code={install_result.returncode})")
            if install_result.stderr:
                for line in install_result.stderr.split('\n')[-30:]:
                    if line.strip():
                        print(f"  {line}")
            if required:
                gs_extensions_ok = False

    probe = subprocess.run(
        [sys.executable, "-c", "import diff_gaussian_rasterization, simple_knn._C; print('ok')"],
        text=True,
        capture_output=True,
        cwd=str(gs_path),
    )
    if probe.returncode != 0 or "ok" not in probe.stdout:
        gs_extensions_ok = False
        print("[warn] Probe final das extensões do gs_static falhou.")
        if probe.stderr:
            for line in probe.stderr.split('\n')[-20:]:
                if line.strip():
                    print(f"  {line}")
else:
    print("[info] Repositório gaussian_splatting ausente; pulando compilação de extensões.")
    runtime_deps_ok = False

print("\n" + "=" * 70)
print(f"Repositórios prontos: {', '.join(cloned_methods) if cloned_methods else '[nenhum clonado com sucesso]'}")
print("Extensões do gs_static: " + ("prontas" if gs_extensions_ok else "pendentes"))
print("=" * 70)
if gs_extensions_ok:
    print("\nOK: ambiente pronto.")
else:
    print("\nAVISO: ambiente base pronto, mas gs_static ainda nao esta compilado corretamente.")


In [ ]:
# Montagem opcional do Google Drive
# Mude USE_GOOGLE_DRIVE para False se quiser rodar sem persistência no Drive
print("=" * 70)
print("Configuração: Google Drive")
print("=" * 70)
print(f"Backup no Drive: {'SIM' if USE_GOOGLE_DRIVE else 'NÃO'}")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    from pathlib import Path

    drive.mount("/content/drive")

    Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    Path(DRIVE_DATA_DIR).mkdir(parents=True, exist_ok=True)
    Path(DRIVE_ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)

    print("✓ Drive montado com sucesso.")
    print(f"✓ Cache de dados: {DRIVE_DATA_DIR}")
    print(f"✓ Cache de artefatos: {DRIVE_ARTIFACTS_DIR}")
else:
    print("  (Para ativar, mude USE_GOOGLE_DRIVE = True na Célula 2)")

print()

In [ ]:
# Verificação de GPU no Colab
print("=" * 70)
print("Verificação de Hardware")
print("=" * 70)

import torch
cuda_available = torch.cuda.is_available()
print(f"CUDA disponível: {'SIM ✓' if cuda_available else 'NÃO ✗'}")
if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ GPU não detectada. O benchmark rodará em CPU (mais lento).")
print()

In [ ]:
# Upload opcional de artifacts pré-treinados (skip training + metrics only)
# Marque SKIP_TRAINING = True na Célula 2 se quiser usar esta célula
print("=" * 70)
print("Configuração: Upload de Artifacts (Opcional)")
print("=" * 70)
print(f"Skip training:  {'SIM' if SKIP_TRAINING else 'NÃO'}")

if SKIP_TRAINING:
    print("\n✓ Modo metrics-only ativado.")
    print("\nEstrutura esperada do arquivo ZIP:")
    print("  artifacts.zip")
    print("    ├── checkpoint.pth (ou .pkl, .ckpt, .pt)")
    print("    └── renders/")
    print("        ├── 0000.png")
    print("        ├── 0001.png")
    print("        └── ...")
    print()

    # Importar google.colab.files
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError(
            "google.colab não disponível. Este notebook foi desenhado para Google Colab.\n"
            "Abra em: https://colab.research.google.com/github/PedroHeinrichSP/TCC-Source-Code/blob/update/notebooks/nvs_benchmark_colab.ipynb"
        )

    print("Clique em 'Escolher arquivos' abaixo para fazer upload do ZIP:")
    uploaded = files.upload()

    if uploaded:
        zip_file = list(uploaded.keys())[0]
        zip_path = f"/content/{zip_file}"

        print(f"\n✓ Arquivo carregado: {zip_file}")
        print(f"  Tamanho: {Path(zip_path).stat().st_size / (1024*1024):.1f} MB")

        # Extrair artifacts
        artifacts_extract_dir = "/content/uploaded_artifacts"
        try:
            checkpoint_path, renders_dir = extract_zip_artifacts(zip_path, artifacts_extract_dir)

            # Salvar globais
            UPLOADED_CHECKPOINT = checkpoint_path
            UPLOADED_RENDERS_DIR = renders_dir
            SKIP_TRAINING = True

            print(f"\n✓ Artifacts extraídos com sucesso:")
            print(f"  Checkpoint:  {UPLOADED_CHECKPOINT}")
            print(f"  Renders:     {UPLOADED_RENDERS_DIR}")
            print(f"\nNa próxima célula (Célula 10), métricas serão recomputadas a partir desses artifacts.")
        except Exception as exc:
            print(f"\n✗ Erro ao extrair artifacts: {exc}")
            print("\nVerifique se o ZIP contém:")
            print("  - Um arquivo de checkpoint (*.pth, *.pkl, *.ckpt, *.pt)")
            print("  - Um diretório 'renders/' com arquivos PNG")
            SKIP_TRAINING = False
            UPLOADED_CHECKPOINT = None
            UPLOADED_RENDERS_DIR = None
    else:
        print("\nNenhum arquivo foi carregado.")
        print("Se mudar de ideia, execute esta célula novamente.")
        SKIP_TRAINING = False
        UPLOADED_CHECKPOINT = None
        UPLOADED_RENDERS_DIR = None
else:
    print("  (Para ativar, mude SKIP_TRAINING = True na Célula 2)")
    print("  Modo normal: clone → setup → download datasets → benchmark completo")

print()

In [ ]:
# Download de datasets via catálogo com cache no Drive
# Se USE_GOOGLE_DRIVE=True, tenta restaurar datasets já baixados antes de instalar de novo.
print("=" * 70)
print("Preparando datasets...")
print("=" * 70)
print(f"Dataset selecionado para execução: {SELECTED_DATASET}")
print(f"Root antes do restore: {SELECTED_DATASET_ROOT}")
print()

from pathlib import Path

local_data_dir = Path("./data")
drive_data_dir = Path(DRIVE_DATA_DIR)

cache_restored = False
if USE_GOOGLE_DRIVE:
    if sync_dir_if_exists(drive_data_dir, local_data_dir):
        print(f"✓ Cache de datasets restaurado de: {drive_data_dir}")
        cache_restored = True
    else:
        print("[info] Nenhum cache de datasets encontrado no Drive. Será feita instalação local.")

# Verificar quais datasets estão disponíveis após cache restore
available_for_run = discover_available_datasets(USE_GOOGLE_DRIVE, DRIVE_DATA_DIR)

# DIAGNÓSTICO detalhado
print("\n[diagnóstico] Datasets descobertos:")
if available_for_run:
    for dataset_name, path in available_for_run.items():
        path_obj = Path(path)
        exists = path_obj.exists()
        has_train = (path_obj / "transforms_train.json").exists()
        has_test = (path_obj / "transforms_test.json").exists()
        print(f"  - {dataset_name}: {path}")
        print(f"      exists={exists} | train={has_train} | test={has_test}")
else:
    print("  [nenhum dataset encontrado]")

if SELECTED_DATASET in available_for_run:
    SELECTED_DATASET_ROOT = available_for_run[SELECTED_DATASET]
    # Force-normalize para eliminar qualquer duplicação do cache restaurado
    SELECTED_DATASET_ROOT = str(normalize_dataset_path(Path(SELECTED_DATASET_ROOT)))
    print(f"\n✓ Dataset '{SELECTED_DATASET}' já disponível em: {SELECTED_DATASET_ROOT}")
    print("  (Pulando download automático)")
else:
    print(f"\n⚠ Dataset '{SELECTED_DATASET}' não encontrado. Tentando instalar...")

    install_cmd = [
        sys.executable,
        "-m",
        "nvs_benchmark.cli",
        "install",
        "--catalog-file",
        CATALOG_FILE,
        "--only",
        "datasets",
        "--execute",
    ]

    install_result = run_logged(install_cmd, label="datasets-install", check=False)

    # Validar novamente após install
    available_for_run = discover_available_datasets(USE_GOOGLE_DRIVE, DRIVE_DATA_DIR)
    if SELECTED_DATASET in available_for_run:
        SELECTED_DATASET_ROOT = available_for_run[SELECTED_DATASET]
        print(f"✓ Root selecionado atualizado para: {SELECTED_DATASET_ROOT}")
    else:
        print(f"\n⚠ AVISO: Dataset '{SELECTED_DATASET}' ainda não disponível após install.")
        print("  Você pode:")
        print("  1. Tentar novamente (alguns downloads são intermitentes)")
        print(f"  2. Baixar manualmente (ver URLs em {CATALOG_FILE})")
        print("  3. Selecionar um dataset diferente (veja lista acima)")

# Persistir root escolhido para as próximas células
if "save_selection_state" in globals():
    state_file_path = get_selection_state_file(USE_GOOGLE_DRIVE, DRIVE_SELECTION_STATE_FILE, LOCAL_SELECTION_STATE_FILE)
    current_state_update = {
        "selected_method": SELECTED_METHOD,
        "selected_dataset": SELECTED_DATASET,
        "selected_preset": SELECTED_PRESET,
        "run_mode": RUN_MODE,
        "apply_compatibility_filter": APPLY_COMPATIBILITY_FILTER,
        "strict_results": STRICT_RESULTS,
        "generate_pdf": GENERATE_PDF,
        "min_required_pairs": MIN_REQUIRED_PAIRS,
    }
    save_selection_state(state_file_path, current_state_update, reason="cell9-dataset-root-refresh")

if USE_GOOGLE_DRIVE and local_data_dir.exists():
    sync_dir_if_exists(local_data_dir, drive_data_dir)
    print(f"✓ Cache de datasets sincronizado para: {drive_data_dir}")

print(f"\n[resumo] Root final selecionado: {SELECTED_DATASET_ROOT}")

# ============================================================================
# GERAR RUN_ID AUTOMATICAMENTE
# ============================================================================
RUN_ID = generate_run_id(environment=ENVIRONMENT, preset=SELECTED_PRESET, method=SELECTED_METHOD, dataset=SELECTED_DATASET)
print(f"RUN_ID auto-gerado: {RUN_ID}")
print("=" * 70)
print("✓ Datasets preparados e RUN_ID gerado.")

Preparando datasets...
Dataset selecionado para execução: blender_synthetic
Root antes do restore: C:\Users\Admin\Projetos\TCC\data\blender_synthetic\nerf_synthetic\lego



NameError: name 'DRIVE_DATA_DIR' is not defined

In [ ]:
# Benchmark (usa seleções da Célula 9)
# Detecta automaticamente se é metrics-only ou full training
print("=" * 70)
print(f"Executando benchmark: {SELECTED_METHOD} x {SELECTED_DATASET}")
print("=" * 70)

from pathlib import Path
import json
import os
import torch

snapshot_file = f"./artifacts/metrics/{RUN_ID}.json"

# ============================================================================
# DETECTAR MODO DE EXECUÇÃO
# ============================================================================
metrics_only_mode = SKIP_TRAINING and UPLOADED_CHECKPOINT and UPLOADED_RENDERS_DIR
if metrics_only_mode:
    print(f"\n[modo] METRICS-ONLY (recompute de artifacts pré-treinados)")
    print(f"  Checkpoint:  {UPLOADED_CHECKPOINT}")
    print(f"  Renders:     {UPLOADED_RENDERS_DIR}")
else:
    print(f"\n[modo] FULL TRAINING (clone → train → infer → metrics)")

print()

if metrics_only_mode:
    # ============================================================================
    # MODO METRICS-ONLY: Use o comando metrics-compute
    # ============================================================================
    cmd = [
        sys.executable,
        "-m",
        "nvs_benchmark.cli",
        "metrics-compute",
        "--method",
        SELECTED_METHOD,
        "--checkpoint",
        UPLOADED_CHECKPOINT,
        "--rendered-dir",
        UPLOADED_RENDERS_DIR,
        "--dataset",
        SELECTED_DATASET,
        "--root",
        SELECTED_DATASET_ROOT,
        "--split",
        "train",
        "--snapshot-file",
        snapshot_file,
        "--append-snapshot",
        "--train-seconds",
        str(UPLOADED_TRAIN_SECONDS),
        "--inference-seconds",
        str(UPLOADED_INFERENCE_SECONDS),
        "--log-dir",
        "./logs",
    ]

    if STRICT_RESULTS:
        cmd.extend(["--strict-results", "--min-required-pairs", str(MIN_REQUIRED_PAIRS)])

    print("Executando metrics-compute...")
    result = run_logged(cmd, label="metrics-compute", check=False)
    if result.returncode != 0:
        raise RuntimeError(
            f"metrics-compute falhou para {SELECTED_METHOD} (code={result.returncode}). "
            "Veja os logs acima para a causa exata."
        )
else:
    # ============================================================================
    # MODO FULL TRAINING: Use o comando method-run
    # ============================================================================
    # Forçar visibilidade da GPU no subprocess do treino
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")

    # Validar GPU no mesmo interpretador que executará o método
    cuda_probe_cmd = [
        sys.executable,
        "-c",
        "import torch; print('CUDA_SUBPROCESS=', torch.cuda.is_available()); print('GPU_NAME=', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')",
    ]
    _ = run_logged(cuda_probe_cmd, label="cuda-subprocess-check", check=False)

    # Validação rápida do dataset antes do benchmark pesado
    root_path = Path(SELECTED_DATASET_ROOT)
    print("[preflight-diagnóstico]")
    print(f"  cwd:                   {Path.cwd()}")
    print(f"  selected_dataset:      {SELECTED_DATASET}")
    print(f"  selected_dataset_root: {SELECTED_DATASET_ROOT}")
    print(f"  root_exists:           {root_path.exists()}")
    print(f"  has_transforms_train:  {(root_path / 'transforms_train.json').exists()}")
    print(f"  has_transforms_test:   {(root_path / 'transforms_test.json').exists()}")
    print(f"  cuda_in_notebook:      {torch.cuda.is_available()}")

    if not root_path.exists():
        print("  [recover] tentando redescobrir root do dataset...")
        recovered = discover_available_datasets(USE_GOOGLE_DRIVE, DRIVE_DATA_DIR)
        print(f"  [recover] datasets encontrados: {list(recovered.keys())}")
        if SELECTED_DATASET in recovered:
            SELECTED_DATASET_ROOT = recovered[SELECTED_DATASET]
            SELECTED_DATASET_ROOT = str(normalize_dataset_path(Path(SELECTED_DATASET_ROOT)))
            root_path = Path(SELECTED_DATASET_ROOT)
            print(f"  [recover] root atualizado para: {SELECTED_DATASET_ROOT}")
            if "save_selection_state" in globals():
                state_file_path = get_selection_state_file(USE_GOOGLE_DRIVE, DRIVE_SELECTION_STATE_FILE, LOCAL_SELECTION_STATE_FILE)
                current_state_update = {
                    "selected_method": SELECTED_METHOD,
                    "selected_dataset": SELECTED_DATASET,
                    "selected_preset": SELECTED_PRESET,
                    "run_mode": RUN_MODE,
                    "apply_compatibility_filter": APPLY_COMPATIBILITY_FILTER,
                    "strict_results": STRICT_RESULTS,
                    "generate_pdf": GENERATE_PDF,
                    "min_required_pairs": MIN_REQUIRED_PAIRS,
                }
                save_selection_state(state_file_path, current_state_update, reason="cell10-preflight-recover-root")

    preflight_cmd = [
        sys.executable,
        "-m",
        "nvs_benchmark.cli",
        "dataset-check",
        "--dataset",
        SELECTED_DATASET,
        "--root",
        SELECTED_DATASET_ROOT,
        "--split",
        "train",
    ]
    preflight = run_logged(preflight_cmd, label="dataset-preflight", check=False)
    if preflight.returncode != 0:
        raise RuntimeError(
            f"dataset-check falhou para {SELECTED_DATASET} em {SELECTED_DATASET_ROOT}. "
            "Corrija a raiz do dataset na Célula 8 antes de continuar."
        )

    # Tuning leve para evitar cair em RAM de CPU no Colab
    nerf_extra = {
        "nerf_half_res": True,
        "nerf_n_rand": 256,
        "nerf_n_samples": 32,
        "nerf_n_importance": 0,
        "nerf_chunk": 1024,
        "nerf_netchunk": 4096,
    }

    cmd = [
        sys.executable,
        "-m",
        "nvs_benchmark.cli",
        "method-run",
        "--method",
        SELECTED_METHOD,
        "--dataset",
        SELECTED_DATASET,
        "--root",
        SELECTED_DATASET_ROOT,
        "--split",
        "train",
        "--preset",
        SELECTED_PRESET,
        "--output-dir",
        "./artifacts",
        "--log-dir",
        "./logs",
        "--compute-metrics",
        "--snapshot-file",
        snapshot_file,
        "--append-snapshot",
    ]

    if SELECTED_METHOD.startswith("nerf_"):
        cmd.extend(["--extra-json", json.dumps(nerf_extra)])

    if STRICT_RESULTS:
        cmd.extend(["--strict-results", "--min-required-pairs", str(MIN_REQUIRED_PAIRS)])

    result = run_logged(cmd, label="method-run", check=False)
    if result.returncode != 0:
        combined_output = f"{result.stdout}\n{result.stderr}"
        oom_like = ("exit=-9" in combined_output) or ("out of memory" in combined_output.lower())
        if oom_like and SELECTED_METHOD.startswith("nerf_") and SELECTED_PRESET != "smoke":
            print("\n[warn] Detectado OOM/kill (exit=-9). Tentando fallback automático com preset smoke...")
            retry_cmd = [x for x in cmd]
            preset_idx = retry_cmd.index("--preset") + 1
            retry_cmd[preset_idx] = "smoke"
            retry_result = run_logged(retry_cmd, label="method-run-smoke-fallback", check=False)
            if retry_result.returncode == 0:
                print("✓ Fallback smoke concluído com sucesso.")
            else:
                raise RuntimeError(
                    "method-run falhou e o fallback smoke também falhou. "
                    "Verifique se a GPU está ativa em Runtime > Change runtime type > T4 e reinicie o runtime."
                )
        else:
            raise RuntimeError(
                f"method-run falhou para {SELECTED_METHOD} x {SELECTED_DATASET} (code={result.returncode}). "
                "Veja os logs acima para a causa exata."
            )

print(f"\n✓ Snapshot gerado em: {snapshot_file}")

In [ ]:
# Gerar relatorio HTML (usa seleções da Célula 7)
print("=" * 70)
print("Gerando relatório HTML...")
print("=" * 70)

report_name = f"{RUN_ID}_{SELECTED_METHOD}_{SELECTED_DATASET}_report"

cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "report-generate",
    "--snapshot-file", snapshot_file,
    "--output-dir", "./artifacts/reports",
    "--report-name", report_name,
    "--log-dir", "./logs",
]

if not GENERATE_PDF:
    cmd.append("--no-pdf")

if STRICT_RESULTS:
    cmd.extend(["--strict-snapshot", "--min-methods", "1", "--require-finite-metrics"])

report_result = run_logged(cmd, label="report-generate", check=False)
if report_result.returncode != 0:
    raise RuntimeError("Falha ao gerar o relatório consolidado.")

report_html = f"./artifacts/reports/{report_name}.html"

if USE_GOOGLE_DRIVE:
    sync_dir_if_exists(Path("./artifacts"), Path(DRIVE_ARTIFACTS_DIR))
    print(f"✓ Artefatos sincronizados para: {DRIVE_ARTIFACTS_DIR}")

print(f"\n✓ Relatório HTML gerado: {report_html}")

In [ ]:
# Exibir o relatório no notebook
print("=" * 70)
print("Carregando relatório...")
print("=" * 70)

from IPython.display import IFrame, display
import os

if not os.path.exists(report_html):
    print(f"✗ Arquivo não encontrado: {report_html}")
    print("  Verifique se a célula anterior executou sem erros.")
else:
    print(f"✓ Abrindo: {report_html}\n")
    display(IFrame(src=report_html, width=1200, height=700))

In [ ]:
# Compactar e baixar artefatos
print("=" * 70)
print("Preparando download de artefatos...")
print("=" * 70)

from google.colab import files
from pathlib import Path
import shutil

zip_path = "/content/nvs_benchmark_artifacts"
print(f"Compactando {REPO_DIR}/artifacts...")
archive_file = shutil.make_archive(zip_path, "zip", REPO_DIR, "artifacts")
print(f"Arquivo gerado: {archive_file}")

if USE_GOOGLE_DRIVE:
    drive_archive_dir = Path(DRIVE_ARTIFACTS_DIR) / "archives"
    drive_archive_dir.mkdir(parents=True, exist_ok=True)
    drive_archive_file = drive_archive_dir / Path(archive_file).name
    shutil.copy2(archive_file, drive_archive_file)
    print(f"✓ ZIP salvo no Drive em: {drive_archive_file}")

if Path(archive_file).exists():
    size_mb = Path(archive_file).stat().st_size / (1024*1024)
    print(f"Tamanho: {size_mb:.1f} MB")
    print("\n↓ Iniciando download...")
    files.download(archive_file)
else:
    print("✗ Arquivo não encontrado!")

In [ ]:
# Backup final dos resultados no Google Drive
print("=" * 70)
print("Backup de Resultados")
print("=" * 70)

if USE_GOOGLE_DRIVE:
    print("Sincronizando resultados com Google Drive...")
    
    # Métricas
    metrics_src = Path("./artifacts/metrics")
    metrics_dst = Path(DRIVE_ARTIFACTS_DIR) / "metrics"
    if metrics_src.exists():
        if metrics_dst.exists():
            shutil.rmtree(metrics_dst)
        shutil.copytree(metrics_src, metrics_dst)
        print(f"  ✓ Métricas → {metrics_dst}")
    
    # Relatórios
    reports_src = Path("./artifacts/reports")
    reports_dst = Path(DRIVE_ARTIFACTS_DIR) / "reports"
    if reports_src.exists():
        if reports_dst.exists():
            shutil.rmtree(reports_dst)
        shutil.copytree(reports_src, reports_dst)
        print(f"  ✓ Relatórios → {reports_dst}")
    
    # Datasets (se usar cache)
    if Path("./data").exists():
        drive_data_dst = Path(DRIVE_DATA_DIR)
        if drive_data_dst.exists():
            shutil.rmtree(drive_data_dst)
        shutil.copytree("./data", drive_data_dst)
        print(f"  ✓ Datasets → {drive_data_dst}")
    
    print(f"\n✓ Backup concluído em: {DRIVE_OUTPUT_DIR}")
else:
    print("Backup: DESATIVADO")
    print("  Para ativar, mude USE_GOOGLE_DRIVE = True na Célula 2")